# ECT token regularisers: condition dropout vs noise (Colab, resumable)

**Phase 1, Cells 6–8:** concat three ways, all with PONITA's official MD17 setup (`--official`, 5000 epochs):
- Cell 6, no regulariser: plain concat, the reference
- Cell 7, **condition dropout**, `--cond-drop 0.15`. For each molecule, with probability 0.15, the ECT token is replaced by a learned null token, so the backbone also has to predict energies and forces without it.
- Cell 8, **token noise**, `--cond-noise 0.5`. Gaussian noise is added to the token at a level drawn per molecule from U(0, 0.5). The level is measured in units of each feature's spread across the batch, i.e. the part that tells conformations apart.

Both are on only during training. Validation, checkpoints and test use the clean token.

**Phase 2, Cells 9–12:** set `WINNER = 'drop'` or `'noise'` in Cell 2. Each cell then trains one model with it: film-final, leap, hybrid-adaln and hybrid-leap.
Cell 14 suggests the winner by **validation** force error, so the test set plays no part in the choice.
The regulariser goes on every ECT input: the global token, or the per-atom local ECTs for leap, and both for hybrid-leap.
For per-atom ECTs, the noise spread is taken over all atoms, so the same σ is a stronger perturbation there.

**Results, Cells 13–14:** curves, and a table comparing each run with the same model without a regulariser (`Δ vs none`) when that run is in the checkpoint folder.
For dropout runs the table also shows the test MAE with the null token for every molecule. If that is close to the normal MAE, the model hardly uses ECT.

Run the cells one at a time, not "Run all". Every training cell resumes from its last checkpoint, and a finished run's cell
just says so. If Colab disconnects, run Cells 1–3 again, then the cell that was running.

Checkpoints go to `dino_ckpts_official` on Drive, the same folder as `colab_official.ipynb`. The plain concat run (Cell 6),
the baseline, and the plain phase-2 models are reused from there, not retrained. Runs with the new flags get their own file names
(`..._cdrop015`, `..._cnoise05`), so nothing existing is overwritten.

### One-time setup
The repo on Drive (`MyDrive/dino`) must have the new `train_ponita_md17.py` and `models/ponita_ect.py`
(with `--cond-drop` / `--cond-noise`), plus `models/egnn_ect.py` and `qm9_ect.py`. Cell 4 checks this.
rMD17 aspirin must be set up as for `colab_official.ipynb` (Cell 5 reuses it).

In [ ]:
# Cell 1 — confirm you have a GPU
!nvidia-smi

In [ ]:
# Cell 2 — mount Drive, paths and settings (re-run after any reconnect, and after setting WINNER)
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = '/content/drive/MyDrive/dino'
CKPT_DIR = '/content/drive/MyDrive/dino_ckpts_official'   # shared with colab_official.ipynb
DATA_DIR = '/content/drive/MyDrive/dino_data/md17'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
assert os.path.exists(os.path.join(REPO_DIR, 'train_ponita_md17.py')), \
    f'train_ponita_md17.py not found in {REPO_DIR} — fix REPO_DIR'

EPOCHS = 5000
SEED   = 0
DROP   = 0.15   # --cond-drop: chance per molecule that its ECT token is swapped for a learned null token
NOISE  = 0.5    # --cond-noise: noise level per molecule ~ U(0, NOISE), in units of the token's spread over the batch
WINNER = None   # after phase 1: 'drop' or 'noise' (Cell 14 suggests one). The phase-2 cells train with it.

REG_ARGS = {'none': '', 'drop': f'--cond-drop {DROP}', 'noise': f'--cond-noise {NOISE}'}
REG = REG_ARGS.get(WINNER)     # the phase-2 cells' regulariser flag
PHASE2_MODELS = ['film-final', 'leap', 'hybrid-adaln', 'hybrid-leap']
MODELS = ['concat'] + (PHASE2_MODELS if WINNER else [])      # rows of the curves and the table

# How runs are named on disk (mirrors _config_tag in train_ponita_md17.py).
MOL    = 'revised_aspirin'
SUFFIX = '_mse_sum_aug_ema099_loop_frms'   # what --official adds
TAGS   = {'none': '', 'drop': f'_cdrop{DROP:g}'.replace('.', ''), 'noise': f'_cnoise{NOISE:g}'.replace('.', '')}

def stem(model, reg):
    return f'{CKPT_DIR}/ponita_PONITA_p_ECT_{model}_{MOL}_seed{SEED}{SUFFIX}{TAGS[reg]}'

BASELINE = f'{CKPT_DIR}/ponita_PONITA_baseline_{MOL}_seed{SEED}{SUFFIX}'
COMMON = (f'--molecule revised-aspirin --official --epochs {EPOCHS} --seed {SEED} '
          f'--data-root "{DATA_DIR}" --ckpt-dir "{CKPT_DIR}" '
          '--log-interval 25 --warmup 100 --batch-size 5 --lr 5e-4 '
          '--lambda-f 500 --hidden-dim 128 --num-layers 5 --num-ori 20 '
          '--basis-dim 256 --ponita-radius 1000.0 --ect-out-dim 64 --icosphere-nu 3')

def load(path):
    import json
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)

def start(model, reg):
    """Log file for a run, or None (with a message) if there is nothing to train."""
    if reg is None:
        print(f'{model}: set WINNER in Cell 2 first (phase 2).')
        return None
    if os.path.exists(stem(model, reg) + '_results.json'):
        print(f'{model} [{reg}]: finished; see Cell 14.')
        return None
    os.chdir(REPO_DIR)
    return stem(model, reg) + '.log'

print('WINNER:', WINNER)

In [ ]:
# Cell 3 — install dependencies matching Colab's torch build
import torch
v = torch.__version__.split('+')[0]
cu = ('cu' + torch.version.cuda.replace('.', '')) if torch.version.cuda else 'cpu'
wheel = f'https://data.pyg.org/whl/torch-{v}+{cu}.html'
print('torch', torch.__version__, '->', wheel)
!pip -q install pyg-lib -f {wheel}
!pip -q install torch-scatter -f {wheel}
!pip -q install torch-geometric
!pip -q install git+https://github.com/ebekkers/ponita.git

In [ ]:
# Cell 4 — sanity check: imports work, the ECT is rotation-invariant, and the code has the new flags
os.chdir(REPO_DIR)
!python test_equivariance.py
import subprocess
res = subprocess.run(['python', 'train_ponita_md17.py', '--help'], capture_output=True, text=True)
assert '--cond-drop' in res.stdout and '--cond-noise' in res.stdout, (
    'This copy of the code has no --cond-drop / --cond-noise. Upload the new train_ponita_md17.py '
    'and models/ponita_ect.py.\n' + res.stderr[-2000:])
print('Code has --cond-drop and --cond-noise.')

In [ ]:
# Cell 5 — rMD17 aspirin where PyG reads it (once; cached on Drive afterwards). Same as colab_official.ipynb.
import glob, shutil
from torch_geometric.datasets import MD17

expected = os.path.join(DATA_DIR, 'raw', 'rmd17', 'npz_data', 'rmd17_aspirin.npz')   # PyG only looks here
if not os.path.exists(expected):
    found = sorted(glob.glob('/content/drive/MyDrive/rmd17*/**/rmd17_aspirin.npz', recursive=True)
                   + glob.glob('/content/drive/MyDrive/rmd17_aspirin.npz'))
    if not found:
        raise FileNotFoundError('rmd17_aspirin.npz not found under MyDrive/rmd17*. Upload it, then re-run.')
    os.makedirs(os.path.dirname(expected), exist_ok=True)
    shutil.copy(found[0], expected)
    print('Copied', found[0], '->', expected)
ds = MD17(root=DATA_DIR, name='revised aspirin')
print(f'rMD17 aspirin ready: {len(ds)} frames')

## Phase 1: concat with each regulariser (Cells 6–8)

In [ ]:
# Cell 6 — concat, no regulariser: the reference. Resumes automatically.
if LOG := start('concat', 'none'):
    !python -u train_ponita_md17.py --model ect --cond-type concat {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 7 — concat + condition dropout. Resumes automatically.
if LOG := start('concat', 'drop'):
    !python -u train_ponita_md17.py --model ect --cond-type concat --cond-drop {DROP} {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 8 — concat + token noise. Resumes automatically.
if LOG := start('concat', 'noise'):
    !python -u train_ponita_md17.py --model ect --cond-type concat --cond-noise {NOISE} {COMMON} 2>&1 | tee -a "{LOG}"

## Phase 2: the other models with the winner (Cells 9–12)
Set `WINNER` in Cell 2 to `'drop'` or `'noise'` (Cell 14 suggests one) and re-run Cell 2. Until then these cells do nothing.

In [ ]:
# Cell 9 — film-final + WINNER (global ECT, FiLM on the final embedding). Resumes automatically.
if LOG := start('film-final', WINNER):
    !python -u train_ponita_md17.py --model ect --cond-type film-final {REG} {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 10 — leap + WINNER (per-atom local ECT at the input). Resumes automatically.
if LOG := start('leap', WINNER):
    !python -u train_ponita_md17.py --model ect --cond-type leap {REG} {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 11 — hybrid-adaln + WINNER (AdaLN in every block, global ECT before the head). Resumes automatically.
if LOG := start('hybrid-adaln', WINNER):
    !python -u train_ponita_md17.py --model ect --cond-type hybrid-adaln {REG} {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 12 — hybrid-leap + WINNER (local ECT at the input, global ECT before the head). Resumes automatically.
if LOG := start('hybrid-leap', WINNER):
    !python -u train_ponita_md17.py --model ect --cond-type hybrid-leap {REG} {COMMON} 2>&1 | tee -a "{LOG}"

## Results (Cells 13–14, re-run any time)

In [ ]:
# Cell 13 — training curves and progress
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Colour = regulariser, the same in every panel. Grey dashed = PONITA baseline, for reference.
STYLE = {
    'none':  dict(color='#2a78d6', label='no regulariser'),
    'drop':  dict(color='#eb6834', label=f'condition dropout (p = {DROP:g})'),
    'noise': dict(color='#1baf7a', label=f'token noise (σ ≤ {NOISE:g})'),
}
BASE_STYLE = dict(color='#8a8880', ls='--', label='PONITA baseline')

def smooth(y, k=25):
    # Train MAE is logged every epoch and is noisy; show a 25-epoch moving average.
    y = np.asarray(y, dtype=float)
    return y if len(y) < k else np.convolve(y, np.ones(k) / k, mode='valid')

def log_tick(v, _):
    # Plain numbers on the log axis, labelling only 1, 2, 3 and 5 of each decade.
    return f'{v:g}' if v > 0 and round(v / 10 ** np.floor(np.log10(v))) in (1, 2, 3, 5) else ''

def draw(ax, h, key, **style):
    if h is None:
        return
    if key.startswith('train'):
        y = smooth(h[key])
        x = np.asarray(h['epoch'][len(h['epoch']) - len(y):])
    else:
        x, y = np.asarray(h['val_epoch']), np.asarray(h[key])
    if len(y):
        ax.plot(x, y, lw=1.8, **style)

panels = [('train_f', 'train force MAE (meV/Å), 25-epoch avg.'),   # measured with dropout / noise on
          ('val_f',   'validation force MAE (meV/Å)'),
          ('val_e',   'validation energy MAE (meV)')]
base = load(BASELINE + '_history.json')
fig, axes = plt.subplots(len(MODELS), 3, figsize=(15, 3.8 * len(MODELS) + 0.4), sharex=True, squeeze=False)
for row, model in zip(axes, MODELS):
    for ax, (key, title) in zip(row, panels):
        draw(ax, base, key, **BASE_STYLE)
        for reg, style in STYLE.items():
            draw(ax, load(stem(model, reg) + '_history.json'), key, **style)
        ax.set_title(f'{model}: {title}', fontsize=10, loc='left')
        ax.set_yscale('log')
        ax.yaxis.set_major_formatter(FuncFormatter(log_tick))
        ax.yaxis.set_minor_formatter(FuncFormatter(log_tick))
        ax.grid(True, which='both', color='#e4e2dc', lw=0.6)
        ax.set_axisbelow(True)
        for side in ('top', 'right'):
            ax.spines[side].set_visible(False)
for ax in axes[-1]:
    ax.set_xlabel('epoch')
legend = {}
for ax in axes.flat:
    for handle, label in zip(*ax.get_legend_handles_labels()):
        legend.setdefault(label, handle)
if legend:
    axes[0, -1].legend(legend.values(), legend.keys(), loc='upper right', frameon=False, fontsize=9)
fig.suptitle(f'ECT token regularisers, rMD17 aspirin, official setup, seed {SEED}',
             x=0.01, ha='left', fontsize=13)
fig.tight_layout()
out = f'{CKPT_DIR}/cond_reg_curves_seed{SEED}.png'
fig.savefig(out, dpi=150)
plt.show()
print('Saved', out)

runs = [('concat', r) for r in ('none', 'drop', 'noise')] + [(m, WINNER) for m in PHASE2_MODELS if WINNER]
for model, reg in runs:
    h = load(stem(model, reg) + '_history.json')
    status = ('finished' if os.path.exists(stem(model, reg) + '_results.json') else
              f"{len(h['epoch'])}/{EPOCHS} epochs" if h else 'not started')
    print(f'  {model:<13} {reg:<6} {status}')

In [ ]:
# Cell 14 — validation and test MAE at the best-force and best-energy checkpoints (written when a run finishes)
import numpy as np
import pandas as pd

# "null token" columns (dropout runs only): the same weights with every molecule's ECT token
# replaced by the learned null token. Close to the normal E / F = the model barely uses the token.
REG_NAME = {'none': 'none', 'drop': f'dropout {DROP:g}', 'noise': f'noise {NOISE:g}'}
rows = []

def add(model, reg, path):
    r = load(path + '_results.json') or {}
    for which, ckpt in (('best_f', 'best val F'), ('best_e', 'best val E')):
        if which in r:
            x = r[which]
            rows.append({'model': model, 'regulariser': REG_NAME[reg], 'checkpoint': ckpt,
                         'val E': x['val_e'], 'val F': x['val_f'],
                         'E (meV)': x['test_e'], 'F (meV/Å)': x['test_f'],
                         'E, null token': x.get('test_e_null', np.nan),
                         'F, null token': x.get('test_f_null', np.nan)})

add('baseline', 'none', BASELINE)
for model in MODELS:
    for reg in REG_NAME:
        add(model, reg, stem(model, reg))
if not rows:
    print('No *_results.json yet; a run writes one when it finishes.')
else:
    df = pd.DataFrame(rows)
    plain = df[df.regulariser == 'none'].set_index(['model', 'checkpoint'])
    for col, delta in (('E (meV)', 'Δ E vs none'), ('F (meV/Å)', 'Δ F vs none')):
        df[delta] = [r[col] / plain.loc[(r.model, r.checkpoint), col] - 1
                     if r.regulariser != 'none' and (r.model, r.checkpoint) in plain.index else np.nan
                     for _, r in df.iterrows()]
    df['model'] = pd.Categorical(df.model, ['baseline'] + MODELS)
    df = df.sort_values(['checkpoint', 'model'], ascending=[False, True], kind='stable').reset_index(drop=True)
    cols = ['model', 'regulariser', 'checkpoint', 'val E', 'val F', 'E (meV)', 'F (meV/Å)',
            'Δ E vs none', 'Δ F vs none', 'E, null token', 'F, null token']
    fmt = {'val E': '{:.2f}', 'val F': '{:.2f}', 'E (meV)': '{:.2f}', 'F (meV/Å)': '{:.2f}',
           'Δ E vs none': '{:+.1%}', 'Δ F vs none': '{:+.1%}',
           'E, null token': '{:.2f}', 'F, null token': '{:.2f}'}
    display(df[cols].style.format(fmt, na_rep='–').hide(axis='index'))
    df[cols].to_csv(f'{CKPT_DIR}/cond_reg_results_seed{SEED}.csv', index=False)

    if WINNER is None:
        # Choose on validation, so the test set stays untouched by the choice.
        f = df[(df.model == 'concat') & (df.checkpoint == 'best val F')].set_index('regulariser')['val F']
        got = {reg: f[REG_NAME[reg]] for reg in ('drop', 'noise') if REG_NAME[reg] in f.index}
        if len(got) == 2:
            win = min(got, key=got.get)
            ref = f", none {f['none']:.2f}" if 'none' in f.index else ''
            print(f"Lower validation force MAE (best-F checkpoint): {win} "
                  f"(dropout {got['drop']:.2f}, noise {got['noise']:.2f}{ref} meV/Å).")
            print(f"This is one seed: check the energy columns and the curves too, then set WINNER = '{win}' in Cell 2.")
        else:
            print('Both regularised concat runs (Cells 7–8) need to finish before a winner can be suggested.')